# Лабораторная №2. Классификация табличных данных с Keras

**Методичка А. А. Друки.** Последовательно выполняются все шесть этапов учебного примера Body Performance, включая все семь фрагментов «Допишите код», а затем самостоятельное задание **вариант 4 — Smoke Detection**.

Датасеты: [Body Performance](https://www.kaggle.com/datasets/kukuroo3/body-performance-data), [Smoke Detection](https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset). В Colab загрузите соответствующие CSV, если они отсутствуют. Метрики появятся только после реального выполнения ячеек.

## 1. Загрузка данных: Body Performance

Изучаем Pandas, scikit-learn, кодирование категорий, разбиение train/validation/test и стандартизацию признаков. Целевая метка `class` имеет четыре значения: A, B, C, D.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
tf.keras.utils.set_random_seed(42)
if not Path("bodyPerformance.csv").exists():
    print("Загрузите bodyPerformance.csv с Kaggle")
    uploaded=files.upload()
    assert "bodyPerformance.csv" in uploaded
data=pd.read_csv("bodyPerformance.csv")
display(data.head())
print(data.shape)
print(data.isna().sum())

### 1.1. Выделение признаков и кодирование пола (пример методички)

In [ ]:
X=data.drop("class",axis=1).copy()
y=data["class"].copy()
label_encoder=LabelEncoder()
X["gender"]=label_encoder.fit_transform(X["gender"])
print("gender:",dict(zip(label_encoder.classes_,label_encoder.transform(label_encoder.classes_))))

### 1.2. Допишите код №1: кодирование целевой переменной

Для `sparse_categorical_crossentropy` классы должны быть целыми индексами **0–3**, а не 1–4. Кодируем отдельным `LabelEncoder`.

In [ ]:
target_encoder=LabelEncoder()
y=target_encoder.fit_transform(y)
print("Классы:",dict(zip(target_encoder.classes_,range(len(target_encoder.classes_)))))
print("Пример меток:",y[:10])

### 1.3. Допишите код №2: обучение, валидация и тестирование

Сначала отделяем 20% тестовых данных, затем 20% оставшихся — на валидацию. Стратификация сохраняет доли классов.

In [ ]:
X_trainval,X_test,y_trainval,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.2,random_state=42,stratify=y_trainval)
print("train:",X_train.shape,"val:",X_val.shape,"test:",X_test.shape)

### 1.4. Допишите код №3: StandardScaler

Среднее и стандартное отклонение вычисляем **только на обучающей выборке**, чтобы избежать утечки данных.

In [ ]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_val_scaled=scaler.transform(X_val)
X_test_scaled=scaler.transform(X_test)
print("Среднее train:",np.round(X_train_scaled.mean(axis=0),3))
print("Стандартное отклонение train:",np.round(X_train_scaled.std(axis=0),3))

## 2. Реализация топологии нейронной сети

### Допишите код №4: Sequential + Dense

Полносвязная сеть с несколькими слоями и `softmax` на выходе для четырёх классов.

In [ ]:
model=Sequential([
    keras.Input(shape=(X_train_scaled.shape[1],)),
    Dense(64,activation="relu"),
    Dense(32,activation="relu"),
    Dense(4,activation="softmax")
])
model.summary()

## 3. Подготовка к обучению

### Допишите код №5: compile

Как указано в методичке: `Adam`, `sparse_categorical_crossentropy`, `accuracy`.

In [ ]:
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

## 4. Обучение нейронной сети

### Допишите код №6: fit

Используем отдельную валидационную выборку. Число эпох можно увеличить для эксперимента.

In [ ]:
history=model.fit(X_train_scaled,y_train,validation_data=(X_val_scaled,y_val),epochs=30,batch_size=32,verbose=1)

## 5. Тестирование нейронной сети

### Допишите код №7: evaluate

Тестовая выборка не использовалась ни при нормализации, ни при обучении.

In [ ]:
loss,accuracy=model.evaluate(X_test_scaled,y_test,verbose=0)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

## 6. Визуализация результатов (пример методички)

Строим графики ошибки и точности для обучения и валидации.

In [ ]:
plt.figure(figsize=(18,6))
plt.subplot(1,3,1)
plt.plot(history.history["loss"],label="Train Loss")
plt.plot(history.history["val_loss"],label="Validation Loss")
plt.xlabel("Epochs");plt.ylabel("Loss");plt.title("Body Performance Loss");plt.legend()
plt.subplot(1,3,2)
plt.plot(history.history["accuracy"],label="Train Accuracy")
plt.plot(history.history["val_accuracy"],label="Validation Accuracy")
plt.xlabel("Epochs");plt.ylabel("Accuracy");plt.title("Body Performance Accuracy");plt.legend()
plt.tight_layout();plt.show()
pred=np.argmax(model.predict(X_test_scaled,verbose=0),axis=1)
print(classification_report(y_test,pred,target_names=target_encoder.classes_))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=target_encoder.classes_,cmap="Blues")
plt.show()

## 7. Самостоятельное задание: вариант 4 — обнаружение пожара

Повторяем **все этапы** учебного примера для датасета Smoke Detection. Целевая переменная в CSV называется `Fire Alarm`, её нормализуем к `Fire_Alarm`. Убираем служебные столбцы `UTC`, `CNT` и индекс, а также предотвращаем утечку данных при масштабировании.

In [ ]:
if Path("smoke_detection_iot.csv").exists():
    smoke_path="smoke_detection_iot.csv"
else:
    uploaded=files.upload()
    candidates=[k for k in uploaded if k.lower().endswith(".csv")]
    assert candidates,"Загрузите smoke_detection_iot.csv"
    smoke_path=candidates[0]
smoke=pd.read_csv(smoke_path)
smoke.columns=[c.strip() for c in smoke.columns]
smoke=smoke.rename(columns={"Fire Alarm":"Fire_Alarm"})
assert "Fire_Alarm" in smoke.columns,smoke.columns.tolist()
display(smoke.head())
print(smoke.shape,smoke["Fire_Alarm"].value_counts())

### 7.1. Предобработка данных и разделение выборок

Выбираем числовые признаки датчиков. Колонки с точками (например, `PM1.0`) корректно обрабатываются Pandas и не требуют переименования, в отличие от PySpark.

In [ ]:
exclude={"Fire_Alarm","UTC","CNT","Unnamed: 0","_c0"}
smoke_X=smoke.drop(columns=[c for c in exclude if c in smoke.columns])
smoke_X=smoke_X.select_dtypes(include=["number"]).replace([np.inf,-np.inf],np.nan)
smoke_y=smoke["Fire_Alarm"].astype(int)
sx_trainval,sx_test,sy_trainval,sy_test=train_test_split(smoke_X,smoke_y,test_size=.2,stratify=smoke_y,random_state=42)
sx_train,sx_val,sy_train,sy_val=train_test_split(sx_trainval,sy_trainval,test_size=.2,stratify=sy_trainval,random_state=42)
from sklearn.impute import SimpleImputer
imp=SimpleImputer(strategy="median")
sx_train=imp.fit_transform(sx_train)
sx_val=imp.transform(sx_val)
sx_test=imp.transform(sx_test)
smoke_scaler=StandardScaler()
sx_train=smoke_scaler.fit_transform(sx_train)
sx_val=smoke_scaler.transform(sx_val)
sx_test=smoke_scaler.transform(sx_test)
print("train/val/test:",len(sx_train),len(sx_val),len(sx_test),"features:",sx_train.shape[1])

### 7.2. Построение и компиляция модели

Два скрытых полносвязных слоя и один выходной нейрон с `sigmoid`. Для бинарной классификации используется `binary_crossentropy`.

In [ ]:
smoke_model=Sequential([
    keras.Input(shape=(sx_train.shape[1],)),
    Dense(64,activation="relu"),
    Dense(32,activation="relu"),
    Dense(1,activation="sigmoid")
])
smoke_model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy",keras.metrics.Precision(name="precision"),keras.metrics.Recall(name="recall")])
smoke_model.summary()

### 7.3. Обучение и тестирование

In [ ]:
smoke_history=smoke_model.fit(sx_train,sy_train,validation_data=(sx_val,sy_val),epochs=25,batch_size=64,verbose=1)
smoke_metrics=smoke_model.evaluate(sx_test,sy_test,verbose=0,return_dict=True)
print("Тестовые метрики:",smoke_metrics)

### 7.4. Графики, матрица ошибок и отчёт

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4))
for i,key in enumerate(["loss","accuracy"]):
    ax[i].plot(smoke_history.history[key],label="Train")
    ax[i].plot(smoke_history.history["val_"+key],label="Validation")
    ax[i].set(title=key,xlabel="Epoch",ylabel=key);ax[i].legend()
plt.tight_layout();plt.show()
smoke_prob=smoke_model.predict(sx_test,verbose=0).ravel()
smoke_pred=(smoke_prob>=0.5).astype(int)
print(classification_report(sy_test,smoke_pred,labels=[0,1],target_names=["Нет пожара","Пожар"],zero_division=0))
ConfusionMatrixDisplay.from_predictions(sy_test,smoke_pred,labels=[0,1],display_labels=["Нет пожара","Пожар"])
plt.show()

## 8. Выводы

После выполнения укажите реальные значения accuracy и loss для четырёхклассовой Body Performance и бинарной Smoke Detection, оцените расхождение train/validation, число пропущенных пожаров (FN) и ложных тревог (FP), а также влияние нормализации. **Метрики заранее не придуманы.**

**Замечание по методичке:** её описание ошибочно называет метки классов после LabelEncoder значениями 1–4, но фактически кодирование начинается с 0; здесь используется корректный диапазон 0–3.